# Llama Embedder Performance and Distance Benchmarks

This notebook benchmarks a llama.cpp embeddings endpoint on:

1. Long **music texts** built from your song **description + lyrics** payloads (`*.text.json`).
2. Long **general texts** (each at least **4096 characters**).

It measures:
- Embedding latency and throughput across batch sizes.
- Pairwise cosine distances.
- Distance behavior for "likely similar" vs "likely different" song pairs.


In [ ]:
# Setup: imports and experiment configuration
from __future__ import annotations

from pathlib import Path
from urllib import request, error
import itertools
import json
import math
import statistics
import time

import matplotlib.pyplot as plt

BASE_URL = "http://127.0.0.1:9002"  # llama.cpp server base URL
MODEL = ""  # optional model name; keep "" to use server default
TIMEOUT_SECONDS = 120

# Music payload discovery (generated by navidrome-embedder text generation pass)
TEXT_PAYLOAD_GLOBS = [
    "../../../../../../../../mnt/data/share/hosted/embeddings/**/*.text.json",
    "output/**/*.text.json",
]

MAX_MUSIC_SONGS = 8
MIN_MUSIC_TEXT_CHARS = 1000

GENERAL_TEXT_MIN_CHARS = 4096
GENERAL_TEXT_TOPICS = [
    "astronomy", "climate", "jazz_history", "software_architecture", "nutrition", "urban_planning"
]

BENCH_BATCH_SIZES = [1, 2, 4, 8]
REPEAT_BENCH_RUNS = 3

EMBEDDINGS_URL = BASE_URL.rstrip("/") + "/v1/embeddings"

print(f"Embeddings endpoint: {EMBEDDINGS_URL}")


Embeddings endpoint: http://127.0.0.1:9002/v1/embeddings


In [6]:
# Helpers: endpoint client, distance math, plotting, and benchmarking

def call_embeddings(texts: list[str], *, dimensions: int | None = None, model: str = MODEL, timeout: float = TIMEOUT_SECONDS):
    payload = {
        "input": texts,
        "encoding_format": "float",
    }
    if model:
        payload["model"] = model
    if dimensions and dimensions > 0:
        payload["dimensions"] = int(dimensions)

    body = json.dumps(payload).encode("utf-8")
    req = request.Request(
        EMBEDDINGS_URL,
        data=body,
        method="POST",
        headers={"Content-Type": "application/json"},
    )

    try:
        with request.urlopen(req, timeout=timeout) as resp:
            data = json.loads(resp.read().decode("utf-8"))
    except error.HTTPError as exc:
        text = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"HTTP {exc.code} from embedding endpoint: {text}") from exc
    except error.URLError as exc:
        raise RuntimeError(f"Could not reach embedding endpoint {EMBEDDINGS_URL}: {exc.reason}") from exc

    if isinstance(data, dict) and isinstance(data.get("error"), dict):
        msg = data["error"].get("message", "unknown error")
        raise RuntimeError(f"Embedding service error: {msg}")

    if isinstance(data, dict) and isinstance(data.get("data"), list):
        entries = []
        for i, row in enumerate(data["data"]):
            if not isinstance(row, dict):
                continue
            emb = row.get("embedding")
            if not isinstance(emb, list):
                continue
            idx = row.get("index", i)
            entries.append((idx, i, [float(v) for v in emb]))
        entries.sort(key=lambda x: (x[0], x[1]))
        return [e[2] for e in entries]

    if isinstance(data, dict) and isinstance(data.get("embedding"), list):
        return [[float(v) for v in data["embedding"]]]

    raise RuntimeError("Embedding response did not contain vectors")


def l2_normalize(vec: list[float]) -> list[float]:
    norm_sq = sum(v * v for v in vec)
    if norm_sq <= 0:
        return vec
    inv = 1.0 / math.sqrt(norm_sq)
    return [v * inv for v in vec]


def cosine_distance(a: list[float], b: list[float]) -> float:
    if len(a) != len(b):
        raise ValueError(f"Dimension mismatch: {len(a)} vs {len(b)}")
    an = l2_normalize(a)
    bn = l2_normalize(b)
    sim = sum(x * y for x, y in zip(an, bn))
    sim = max(-1.0, min(1.0, sim))
    return 1.0 - sim


def pairwise_distance_matrix(vectors: list[list[float]]) -> list[list[float]]:
    n = len(vectors)
    out = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            d = cosine_distance(vectors[i], vectors[j])
            out[i][j] = d
            out[j][i] = d
    return out


def chunked(items: list, size: int):
    for i in range(0, len(items), size):
        yield items[i : i + size]


def benchmark_embedding(texts: list[str], batch_sizes: list[int], runs: int = REPEAT_BENCH_RUNS):
    rows = []
    for batch_size in batch_sizes:
        per_run_secs = []
        for _ in range(runs):
            t0 = time.perf_counter()
            total = 0
            for batch in chunked(texts, batch_size):
                vecs = call_embeddings(batch)
                total += len(vecs)
            dt = time.perf_counter() - t0
            per_run_secs.append(dt)

        mean_sec = statistics.mean(per_run_secs)
        rows.append({
            "batch_size": batch_size,
            "runs": runs,
            "mean_seconds": mean_sec,
            "docs_per_second": (len(texts) / mean_sec) if mean_sec > 0 else float("inf"),
            "mean_ms_per_doc": (mean_sec * 1000.0 / len(texts)) if texts else 0.0,
        })
    return rows


def plot_heatmap(matrix: list[list[float]], labels: list[str], title: str):
    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 0.9), max(6, len(labels) * 0.7)))
    im = ax.imshow(matrix, interpolation="nearest", aspect="auto")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=65, ha="right", fontsize=8)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_title(title)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Cosine distance (lower = more similar)")
    fig.tight_layout()
    plt.show()


In [7]:
# Load music description+lyrics payloads and build long song texts

def discover_text_payloads(globs: list[str]) -> list[Path]:
    root = Path('.').resolve()
    files = {}
    for pat in globs:
        for p in root.glob(pat):
            if p.is_file() and p.name.endswith('.text.json'):
                # avoid pulling tiny metadata files from egg-info or irrelevant dirs
                parts = {x.lower() for x in p.parts}
                if 'navidrome_embedder.egg-info' in parts:
                    continue
                files[str(p)] = p
    return sorted(files.values())


def parse_artist_from_name(name: str) -> str:
    if ' - ' in name:
        return name.split(' - ', 1)[0].strip()
    return 'unknown'


payload_files = discover_text_payloads(TEXT_PAYLOAD_GLOBS)
records = []
for p in payload_files:
    try:
        obj = json.loads(p.read_text(encoding='utf-8'))
    except Exception:
        continue
    name = str(obj.get('name') or obj.get('track_id') or p.stem)
    description = str(obj.get('description') or '').strip()
    lyrics = str(obj.get('lyrics') or '').strip()
    if not description and not lyrics:
        continue

    combined = ''
    if description:
        combined += '[DESCRIPTION]\n' + description + '\n\n'
    if lyrics:
        combined += '[LYRICS]\n' + lyrics

    records.append({
        'name': name,
        'artist': parse_artist_from_name(name),
        'description': description,
        'lyrics': lyrics,
        'combined': combined.strip(),
        'chars': len(combined.strip()),
        'source': str(p),
    })

records.sort(key=lambda r: r['chars'], reverse=True)
long_records = [r for r in records if r['chars'] >= MIN_MUSIC_TEXT_CHARS]
if len(long_records) < MAX_MUSIC_SONGS:
    long_records = records[:MAX_MUSIC_SONGS]

print(f'Discovered payload files: {len(payload_files)}')
print(f'Parsed usable records:   {len(records)}')
print(f'Selected song records:   {len(long_records)}')

if len(long_records) < 4:
    raise RuntimeError(
        'Need at least 4 song text records. Generate *.text.json first using navidrome-embedder text generation pass.'
    )

for r in long_records[:MAX_MUSIC_SONGS]:
    print(f"- {r['name'][:80]} | artist={r['artist'][:40]} | chars={r['chars']}")


NotImplementedError: Non-relative patterns are unsupported

In [ ]:
# Pick songs and define "likely similar" and "likely different" pairs
selected = long_records[:MAX_MUSIC_SONGS]
name_to_idx = {r['name']: i for i, r in enumerate(selected)}

# Optional manual pair overrides (recommended when you know the catalog well)
# Example:
# MANUAL_SIMILAR_PAIRS = [("Artist A - Song 1", "Artist A - Song 2")]
# MANUAL_DIFFERENT_PAIRS = [("Artist A - Song 1", "Artist B - Song X")]
MANUAL_SIMILAR_PAIRS = []
MANUAL_DIFFERENT_PAIRS = []

# Auto similar heuristic: same artist pairs when available
artist_groups = {}
for r in selected:
    artist_groups.setdefault(r['artist'], []).append(r)

auto_similar_pairs = []
for artist, items in artist_groups.items():
    if artist == 'unknown' or len(items) < 2:
        continue
    items = sorted(items, key=lambda x: x['chars'], reverse=True)
    auto_similar_pairs.append((items[0]['name'], items[1]['name']))

# Auto different heuristic: different artists
auto_different_pairs = []
for a, b in itertools.combinations(selected, 2):
    if a['artist'] != b['artist']:
        auto_different_pairs.append((a['name'], b['name']))

similar_pairs = MANUAL_SIMILAR_PAIRS or auto_similar_pairs[:4]
different_pairs = MANUAL_DIFFERENT_PAIRS or auto_different_pairs[:6]

print('Selected songs:')
for i, r in enumerate(selected):
    print(f"{i:>2}: {r['name'][:90]} (artist={r['artist']}, chars={r['chars']})")

print('\nLikely similar pairs:')
for p in similar_pairs:
    print('-', p)

print('\nLikely different pairs:')
for p in different_pairs:
    print('-', p)


In [ ]:
# Embed music texts, benchmark speed, and graph pairwise distances
music_texts = [r['combined'] for r in selected]
music_labels = [r['name'] for r in selected]

# Warmup
_ = call_embeddings([music_texts[0]])

music_vectors = []
for batch in chunked(music_texts, 4):
    music_vectors.extend(call_embeddings(batch))

music_matrix = pairwise_distance_matrix(music_vectors)
plot_heatmap(music_matrix, music_labels, 'Music Text Pairwise Cosine Distance')

# Pair bar chart
pair_labels = []
pair_values = []
pair_colors = []

for a, b in similar_pairs:
    if a in name_to_idx and b in name_to_idx:
        d = music_matrix[name_to_idx[a]][name_to_idx[b]]
        pair_labels.append(f"SIM: {a[:25]} <> {b[:25]}")
        pair_values.append(d)
        pair_colors.append('tab:green')

for a, b in different_pairs:
    if a in name_to_idx and b in name_to_idx:
        d = music_matrix[name_to_idx[a]][name_to_idx[b]]
        pair_labels.append(f"DIFF: {a[:22]} <> {b[:22]}")
        pair_values.append(d)
        pair_colors.append('tab:red')

if pair_labels:
    plt.figure(figsize=(12, max(4, len(pair_labels) * 0.35)))
    y = list(range(len(pair_labels)))
    plt.barh(y, pair_values, color=pair_colors)
    plt.yticks(y, pair_labels, fontsize=8)
    plt.xlabel('Cosine distance (lower = more similar)')
    plt.title('Music Pair Distances: Similar vs Different')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

if similar_pairs:
    sim_vals = [music_matrix[name_to_idx[a]][name_to_idx[b]] for a, b in similar_pairs if a in name_to_idx and b in name_to_idx]
else:
    sim_vals = []

diff_vals = [music_matrix[name_to_idx[a]][name_to_idx[b]] for a, b in different_pairs if a in name_to_idx and b in name_to_idx]

print('Music distance summary:')
if sim_vals:
    print(f"- likely similar mean distance: {statistics.mean(sim_vals):.4f} (n={len(sim_vals)})")
else:
    print('- likely similar mean distance: n/a (no same-artist pair found)')
if diff_vals:
    print(f"- likely different mean distance: {statistics.mean(diff_vals):.4f} (n={len(diff_vals)})")

music_bench = benchmark_embedding(music_texts, BENCH_BATCH_SIZES)
print('\nMusic embedding speed:')
for row in music_bench:
    print(
        f"batch={row['batch_size']:>2} | mean_s={row['mean_seconds']:.3f} | "
        f"docs/s={row['docs_per_second']:.2f} | ms/doc={row['mean_ms_per_doc']:.2f}"
    )


In [ ]:
# Build long general texts (>= 4096 chars each)

GENERAL_PARAGRAPHS = {
    'astronomy': (
        "Astronomy studies stars, planets, interstellar gas, and cosmic structure across scales. "
        "Observation links spectra, luminosity, and dynamics to infer mass, temperature, and age. "
        "Modern surveys combine multi-band imaging with spectroscopy to characterize populations over time. "
    ),
    'climate': (
        "Climate science models atmosphere, ocean, cryosphere, and biosphere interactions under forcing. "
        "Researchers analyze paleoclimate proxies, reanalysis data, and coupled models to estimate sensitivity. "
        "Regional outcomes depend on circulation shifts, extremes, adaptation capacity, and policy timing. "
    ),
    'jazz_history': (
        "Jazz history spans collective improvisation, swing orchestration, bebop complexity, modal exploration, and fusion. "
        "Players shape phrasing through timbre, rhythm section interaction, and harmonic substitution choices. "
        "Recording technology and venue culture influenced arrangement styles and audience listening practices. "
    ),
    'software_architecture': (
        "Software architecture balances modularity, operational reliability, and change velocity under constraints. "
        "Teams define boundaries, contracts, and observability to reduce coupling and improve recovery from faults. "
        "Tradeoffs include latency versus consistency, local simplicity versus global complexity, and tooling costs. "
    ),
    'nutrition': (
        "Nutrition research links dietary patterns, micronutrient adequacy, and metabolic regulation with outcomes. "
        "Evidence quality depends on trial design, confounding control, adherence measurement, and follow-up length. "
        "Recommendations translate uncertainty into practical guidance across cultures, budgets, and health conditions. "
    ),
    'urban_planning': (
        "Urban planning integrates transport, housing, green space, and utilities with demographic and economic trends. "
        "Walkability, transit frequency, and mixed-use zoning influence accessibility and emissions trajectories. "
        "Effective plans align land policy, capital budgets, maintenance, and community participation. "
    ),
}


def make_long_text(topic: str, min_chars: int = GENERAL_TEXT_MIN_CHARS) -> str:
    seed = GENERAL_PARAGRAPHS[topic]
    chunks = [f"Topic: {topic}. "]
    while len(''.join(chunks)) < min_chars:
        chunks.append(seed)
    text = ''.join(chunks)
    return text[: max(min_chars, len(text))]


general_records = []
for topic in GENERAL_TEXT_TOPICS:
    txt = make_long_text(topic, GENERAL_TEXT_MIN_CHARS)
    general_records.append({'topic': topic, 'text': txt, 'chars': len(txt)})

print('General text lengths:')
for r in general_records:
    print(f"- {r['topic']}: {r['chars']} chars")


In [ ]:
# Embed general texts, graph distances, and benchmark speed
general_texts = [r['text'] for r in general_records]
general_labels = [r['topic'] for r in general_records]

# Warmup
_ = call_embeddings([general_texts[0]])

general_vectors = []
for batch in chunked(general_texts, 4):
    general_vectors.extend(call_embeddings(batch))

general_matrix = pairwise_distance_matrix(general_vectors)
plot_heatmap(general_matrix, general_labels, 'General Text (>=4096 chars) Pairwise Cosine Distance')

general_bench = benchmark_embedding(general_texts, BENCH_BATCH_SIZES)
print('General-text embedding speed:')
for row in general_bench:
    print(
        f"batch={row['batch_size']:>2} | mean_s={row['mean_seconds']:.3f} | "
        f"docs/s={row['docs_per_second']:.2f} | ms/doc={row['mean_ms_per_doc']:.2f}"
    )


In [ ]:
# Final summary object for copy/paste or downstream analysis
summary = {
    'endpoint': EMBEDDINGS_URL,
    'model': MODEL or '<server-default>',
    'music': {
        'num_songs': len(selected),
        'benchmarks': music_bench,
        'similar_pairs': similar_pairs,
        'different_pairs': different_pairs,
    },
    'general': {
        'num_texts': len(general_records),
        'min_chars': min(r['chars'] for r in general_records),
        'benchmarks': general_bench,
    },
}
summary


## Notes

- This notebook expects a running llama.cpp embeddings service at `BASE_URL`.
- To start one in this repo, use `scripts/start-qwen-embed-server.sh`.
- If there are not enough local `*.text.json` song payloads, run the text generation stage first with `navidrome-embedder.py`.
